# Practical 2 : Generative and Discriminative Models


In this practical, we will compare the Naïve Bayes Classifier (NBC) and Logistic Regression on several
datasets. As part of the practical you should read briefly the following paper:



**On Discriminative vs. Generative classifiers: A comparison of logistic regression
and naive Bayes**  
*Andrew Y. Ng and Michael I. Jordan*  
Advances in Neural Information Processing Systems (NIPS) 2001.

The paper is available on OLAT. 

You should read the Introduction and the Experiments sections. The goal of this practical is
to qualitatively reproduce some of the experimental results in this paper. You are strongly
encouraged to read the rest of the paper, which is rather short and straightforward to read,
though some of you may want to skip the formal proofs.

## Naïve Bayes Classifier (NBC)

You should implement a Naïve Bayes Classifier from scartch using NumPy. To keep your code tidy,
we recommend implementing it as a class. 
The classifier should be able to handle binary and continuous features. 
To earn the bonus points, the classifier should be able to handle categorical features as well. 
Suppose the data has 3
different features, the first being binary, the second being continuous and the third being categorical. Write an implementation that you can initialise as follows:

    nbc = NBC(feature_types=['b', 'r', 'c'])

Along the lines of classifiers provided in sklearn, you want to implement two more functions,
**fit** and **predict**. 
Recall the joint distribution of a generative model: $p(\mathbf{x}, y \mid \theta, \pi) = p(y \mid \pi) \cdot p(\mathbf{x} \mid y, \theta)$.
The **fit** function is to estimate all the parameters ($\theta$ and $\pi$) of the NBC, i.e., train the classifier. The **predict** function is to compute the probabilities that the new input belongs to all classes and
then return the class that has the largest probability, i.e., make the prediction.

    nbc.fit(X_train, y_train)
    ypredicted = nbc.predict(X_test)
    test_accuracy = np.mean(ypredicted == ytest)

Here we import the libraries. 

In [ ]:
%matplotlib inline
import pylab
pylab.rcParams['figure.figsize'] = (10., 10.)

import pickle as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder


### Class-conditional distributions

Before implementing NBC, we first implement the class-conditional distributions $p(\mathbf{x} \mid y, \theta)$. Your implementation should have two functions: **estimate** and **get_log_probability**. 

- The **estimate** function takes data as input and models the data using some distribution $p(x \mid \theta)$, where $\theta$ is the parameters of this distribution. The function estimates the parameters $\theta$ using maximum likelihood estimators (MLE). 
For example, in the case of continuous features, we use the Gaussian distribution to model the data. The estimate function will find the parameters $\mu$ and $\sigma$ for the Gaussian distribution with respect to the input data. 

- The **get_log_probability** function takes as input a new data point $x_{new}$ and returns the log of $p(x_{new} \mid \theta)$. For the Gaussian distribution, the function get_probability will return $\mathcal{N}(x_{new} \mid \mu, \sigma)$. 

For different types of features, you need to use different distributions.
You can import statistic libraries (e.g., `scipy.stats`) for the implementation of the distributions. 

- For **continuous features**: Use Gaussian distribution
    https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html
- For **binary features**: Use Bernoulli distribution 
    https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bernoulli.html
- For **categorical features**: Use Multinoulli distribution (The multinoulli distribution is a special case of the multinomial distribution, where the number of trials is 1)
    https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.multinomial.html



**Implementation Issues:**
- The probabilities can be very small. To avoid underflow issues, you should compute the log of the probabilities. Read more: (Mur) Chapter 3.5.3 / Lecture Notes
- The standard deviation for Gaussian distributions should never be exactly 0, so in
case your calculated standard deviation is 0, you may want to set it to a small value such as 1e − 6. This is to ensure that your code never encounters division by zero or
taking logarithms of 0 errors. 
For this practical, please set the small value to 1e-6.
- Laplace/Additive smoothing: You want to ensure that the estimates for the parameter for the Bernoulli and Multinoulli random variables is never exactly 0 or 1. For this reason you should consider using Laplace smoothing (https://en.wikipedia.org/wiki/Additive_smoothing).
For this practical, please set alpha to 1.
- We will check the correctness of your implementation using the tests below.
- For simplicity, you can assume the data values for binary features are integers from {0,1} and the data for a categorical feature with M categories are integers from {0, ..., M-1}.
- Fell free to add auxiliary functions or change the parameters of the functions. If you change the parameters of the functions, make sure you change the tests accordingly, so we can test your code.


In [ ]:
from scipy.stats import norm
from scipy.stats import bernoulli
from scipy.stats import multinomial

ALPHA = 1.0 # for additive smoothing

# Distribution for continuous features
class ContFeatureParam:
    def estimate(self, X):
        # TODO: Estimate the parameters for the Gaussian distribution 
        # so that it best describes the input data X
        # The code below is just for compilation. 
        # You need to replace it by your own code.
        ###################################################
        ##### YOUR CODE STARTS HERE #######################
        ###################################################
        self.mean = np.mean(X)
        self.var = np.var(X, ddof=1)
        return self.mean, self.var
        ###################################################
        ##### YOUR CODE ENDS HERE #########################
        ###################################################
        

    def get_log_probability(self, X_new):
        # TODO: return the log of the density values for the input values X_new
        # The code below is just for compilation. 
        # You need to replace it by your own code.
        ###################################################
        ##### YOUR CODE STARTS HERE #######################
        ###################################################
        return norm.logpdf(X_new, self.mean, np.sqrt(self.var))
        ###################################################
        ##### YOUR CODE ENDS HERE #########################
        ###################################################

# Distribution for binary features
class BinFeatureParam:
    def estimate(self, X):
        # TODO: Estimate the parameters for the Bernoulli distribution 
        # so that it best describes the input data X
        # The code below is just for compilation. 
        # You need to replace it by your own code.
        ###################################################
        ##### YOUR CODE STARTS HERE #######################
        ###################################################
        # X is a vector with binary values
        self.p = (np.sum(X) + ALPHA) / (len(X) + 2 *ALPHA)
        return self.p
        ###################################################
        ##### YOUR CODE ENDS HERE #########################
        ###################################################

    def get_log_probability(self, X_new):
        # TODO: return the log of the probability values for the input values X_new
        # The code below is just for compilation. 
        # You need to replace it by your own code.
        ###################################################
        ##### YOUR CODE STARTS HERE #######################
        ###################################################
        return(bernoulli.logpmf(X_new, self.p))
        ###################################################
        ##### YOUR CODE ENDS HERE #########################
        ###################################################

# Distribution for categorical features
class CatFeatureParam:
    
    # we need to know the number of categories for the categorical feature
    def __init__(self, num_of_categories):
        self._num_of_categories = num_of_categories
    
    def estimate(self, X):
        # TODO: Estimate the parameters for the Multinoulli distribution 
        # so that it best describes the input data X
        # The code below is just for compilation. 
        # You need to replace it by your own code.
        ###################################################
        ##### YOUR CODE STARTS HERE #######################
        ###################################################
        counts = np.bincount(X)
        self.p = (counts + ALPHA) / (sum(counts) + self._num_of_categories * ALPHA)
        return self.p
        ###################################################
        ##### YOUR CODE ENDS HERE #########################
        ###################################################
        
    def get_log_probability(self, X_new):
        # TODO: return the log of the probability values for the input values X_new
        # The code below is just for compilation. 
        # You need to replace it by your own code.
        ###################################################
        ##### YOUR CODE STARTS HERE #######################
        ###################################################
        n = np.sum(X_new)
        return multinomial.logpmf(X_new, n, self.p)
        ###################################################
        ##### YOUR CODE ENDS HERE #########################
        ###################################################

**Tests:**
    
We will use the code below to test the correctness of your code.

In [ ]:
# continuous features

X = np.array([2.70508547,2.10499698,1.76019132,3.42016431,3.47037973,3.67435061,1.84749286,4.3388506,2.27818252,4.65165335])

param = ContFeatureParam()
param.estimate(X)
probs = param.get_log_probability(np.array([0,1,2,3]))
print(probs)

In [ ]:
# binary features

X = np.array([0,0,1,1,0,1,0,1,1,1])

param = BinFeatureParam()
param.estimate(X)
probs = param.get_log_probability(np.array([0,1]))
print(probs)

In [ ]:
# categorical features (bonus task)

X = np.array([0,6,5,4,0,6,6,4,1,1,2,3,8,8,1,6,4,9,0,2,2,3,8,0,2])

param = CatFeatureParam(num_of_categories=10)
param.estimate(X)
probs = param.get_log_probability(np.array([0,1,2,3,4,5,6,7,8,9]))
print(probs)

### Implement NBC

We are now ready to implement NBC. We follow the structure of models in scikit-learn. We implement NBC as a class with functions **init**, **fit** and **predict**.
The **init** function takes as input the types of features and initialise the classifier. The **fit** function takes the training data as input and estimates the parameters. The **predict** function predicts the label for the input data. 

**Implementation Issues:**
- You should use matrix operations rather than loops. In general, loops over classes or features are OK, but loops over the rows of data are not a good idea.
- The probabilities can be very small. To avoid underflow issues, you should do the calculations in log space. Read more: (Mur) Chapter 3.5.3 / Lecture Note
- For simplicity, you can assume the data values for binary features are integers from {0, 1} and the data for a categorical feature with M categories are integers from {0, ..., M-1}.
- Fell free to add auxiliary functions or change the parameters of the functions. If you change the parameters of the functions, make sure you change the tests accordingly, so we can test your code.

In [ ]:
# Your task is to implement the three functions of NBC. 
ALPHA = 1.0

class NBC:
    # Inputs:
    #   feature_types: the array of the types of the features, e.g., feature_types=['b', 'r', 'c']
    def __init__(self, feature_types=[]):
        # TODO: 
        # The code below is just for compilation. 
        # You need to replace it by your own code.
        ###################################################
        ##### YOUR CODE STARTS HERE #######################
        ###################################################
        self.feature_types = feature_types
        self.mean = None
        self.var = None
        self.theta = None
        self.cat_probs = None
        self.class_log_prior_ = None
        self.cat_probs = None
        ###################################################
        ##### YOUR CODE ENDS HERE #########################
        ###################################################

        
    # The function uses the input data to estimate all the parameters of the NBC
    def fit(self, X, y):  #X, y = iris['data'], iris['target']
        # TODO: 
        # The code below is just for compilation. 
        # You need to replace it by your own code.
        ###################################################
        ##### YOUR CODE STARTS HERE #######################
        ###################################################
        
        classes = np.unique(y)
        self.classes_ = classes 
        K = len(classes)
        D = X.shape[1]

        self.mean = np.zeros((K, D))
        self.var = np.zeros((K, D))
        self.theta = np.zeros((K, D))
        self.cat_probs = [{} for _ in range(K)]
        self.class_log_prior_ = np.zeros(K)

        for i, c in enumerate(classes):

            X_c = X[y == c]
            N_c = np.sum(y == c)
            P_c = (N_c + ALPHA) / (len(y) + ALPHA * K)
            self.class_log_prior_[i] = np.log(P_c) # P(y)

            for j in range(D):
                if self.feature_types[j] == 'r': # Gaussian
                        column_values = X_c[:, j]
                        mean_cj = np.mean(column_values)
                        var_cj = np.var(column_values)
                        var_cj = max(var_cj, np.finfo(float).eps)
                        self.mean[i][j] = mean_cj
                        self.var[i][j] =var_cj  #P(x_j|y)
                
                elif self.feature_types[j] == 'b': # Bernoulli
                    column_values = (X_c[:, j])
                    N_1 = np.count_nonzero(column_values == 1)
                    N_0 = np.count_nonzero(column_values == 0)
                    
                    # smooth
                    P_1 = (N_1 + ALPHA) / (len(X_c) + 2 * ALPHA)
                    P_0 = (N_0 + ALPHA) / (len(X_c) + 2 * ALPHA)

                    # log
                    # P_1 = np.log(P_1)
                    # P_0 = np.log(P_0)
                    self.theta[i, j] = P_1  #P(x_j|y)

                elif self.feature_types[j] == 'c': # Categorical
                    column_values = X_c[:, j]
                    values = np.unique(X[:, j])
                    counts = np.bincount(column_values, minlength=len(values))
                    P_cjv = (counts + ALPHA) / (len(X_c) + ALPHA * len(values))
                    self.cat_probs[i][j] = P_cjv  #P(x_j|y)




        ###################################################
        ##### YOUR CODE ENDS HERE #########################
        ###################################################
                
                
    # The function takes the data X as input, and predicts the class for the data
    def predict(self, X):
        # TODO: 
        # The code below is just for compilation. 
        # You need to replace it by your own code.
        ###################################################
        ##### YOUR CODE STARTS HERE #######################
        ###################################################
        y_pred = []
        epsilon = 1e-9

        for x in X:
            score = np.zeros(len(self.class_log_prior_))
            for i in range(len(self.class_log_prior_)):
                score[i] = self.class_log_prior_[i]
                for j in range(len(self.feature_types)):
                    if self.feature_types[j] == 'r':
                        x_j = x[j]
                        mean_ij = self.mean[i][j]
                        var_ij = self.var[i][j]
                        log_prob = -0.5 * np.log(2 * np.pi * var_ij) - ((x_j - mean_ij) ** 2) / (2 * var_ij)
                        score[i] += log_prob
                    elif self.feature_types[j] == 'b':
                        x_j = x[j]
                        theta_ij = self.theta[i, j]
                        log_prob = x_j * np.log(theta_ij) + (1 - x_j) * np.log(1 - theta_ij)
                        score[i] += log_prob
                    elif self.feature_types[j] == 'c':
                        x_j = x[j]
                        cat_ij = self.cat_probs[i][j]
                        p = cat_ij[x_j] if x_j < len(cat_ij) else epsilon
                        log_prob = np.log(p)
                        score[i] += log_prob
            pred_class = np.argmax(score)
            y_pred.append(self.classes_[pred_class])
        return np.array(y_pred)


        ###################################################
        ##### YOUR CODE ENDS HERE #########################
        ###################################################


**Tests**

We will use the code below to check your code.

In [ ]:
# All features of the iris dataset are continuous.

from sklearn.datasets import load_iris
iris = load_iris()
X, y = iris['data'], iris['target']
print(X.shape)
print(y.shape)
np.unique(y)

N, D = X.shape
Ntrain = int(0.8 * N)
Xtrain = X[:Ntrain]
ytrain = y[:Ntrain]
Xtest = X[Ntrain:]
ytest = y[Ntrain:]


nbc_iris = NBC(feature_types=['r', 'r', 'r', 'r'])
nbc_iris.fit(Xtrain, ytrain)
yhat = nbc_iris.predict(Xtest)
test_accuracy = np.mean(yhat == ytest)

print("Accuracy:", test_accuracy) # should be larger than 90%
print(yhat)

In [ ]:
# All features of this dataset are binary
import pandas as pd
data = pd.read_csv('./data/binary_test.csv', header=None)
data = data.to_numpy()

X = data[:,1:]
y = data[:,0]
print(X.shape)
print(y.shape)
print(len(y))
np.unique(y)
N, D = X.shape
Ntrain = int(0.8 * N)
Xtrain = X[:Ntrain]
ytrain = y[:Ntrain]
Xtest = X[Ntrain:]
ytest = y[Ntrain:]


nbc = NBC(feature_types=['b'] * 16)
nbc.fit(Xtrain, ytrain)
yhat = nbc.predict(Xtest)
test_accuracy = np.mean(yhat == ytest)

print("Accuracy:", test_accuracy) # should be larger than 85%
print(yhat)

In [ ]:
# All features of this dataset are categorical (bonus task)

# data = pd.read_csv('./datasets/categorical_test.csv', header=None)
data = pd.read_csv('./data/categorical_test.csv', header=None)
data = data.to_numpy()
print(data.shape)
print(data)
X = data[:,:-1]
y = data[:,-1]

N, D = X.shape
Ntrain = int(0.8 * N)
Xtrain = X[:Ntrain]
ytrain = y[:Ntrain]
Xtest = X[Ntrain:]
ytest = y[Ntrain:]


nbc = NBC(feature_types=['c'] * 9)
nbc.fit(Xtrain, ytrain)
yhat = nbc.predict(Xtest)
test_accuracy = np.mean(yhat == ytest)

print("Accuracy:", test_accuracy) # should be larger than 65%
print(yhat) 

## Logistic Regression

For logistic regression, you should use the implementation in scikit-learn. Add the following
line to import the LR model.

In [ ]:
from sklearn.linear_model import LogisticRegression

Check the scikit-learn documentation for the Logistic Regression model:
- http://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
- http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html#sklearn.linear_model.LogisticRegression


## Comparing NBC and LR

### Experiments

The experiment is to compare the classification error of the NBC and LR trained on increasingly larger training datasets. 
Since the datasets are so small, you should do this multiple times and
average the classification error. One run should look as follows:
- Shuffle the data, put 20% aside for testing.
    
    ```N, D = X.shape
    Ntrain = int(0.8 * N)
    shuffler = np.random.permutation(N)
    Xtrain = X[shuffler[:Ntrain]]
    ytrain = y[shuffler[:Ntrain]]
    Xtest = X[shuffler[Ntrain:]]
    ytest = y[shuffler[Ntrain:]]
    
    ```  


- Train the classifiers with increasingly more data. For example, we can train classifiers with 10%, 20%, ..., 100% of the training data. For each case store the classification errors on the test set of the classifiers.

You may want to repeat this with at least 200 random permutations (possibly as large as 1000)
to average out the test error across the runs. In the end, you will get average test errors as a function of the size of the training data. 
We have written for you the function for making the plots for the experiments. 

In [ ]:
# inputs:
#   nbc: Naive Bayes Classifier
#   lr: Logistic Regression Classifier
#   X, y: data
#   num_runs: we need repeat num_runs times and store average results
#   num_splits: we want to compare the two models on increasingly larger training sets.
#               num_splits defines the number of increasing steps. 
# outputs:
#   the arrays of the test errors across the runs of the two classifiers 
def compareNBCvsLR(nbc, lr, X, y, num_runs=200, num_splits=10):
    # The code below is just for compilation. 
    # You need to replace it by your own code.
    ###################################################
    ##### YOUR CODE STARTS HERE #######################
    ###################################################
    tst_errs_nbc = []
    tst_errs_lr = []
    all_nbc_errors = np.zeros((num_runs, num_splits))
    all_lr_errors = np.zeros((num_runs, num_splits))

    N = len(X)
    Ntrain = int(0.8 * N)

    for run in range(num_runs):
        shuffler = np.random.permutation(N)
        Xtrain = X[shuffler[:Ntrain]]
        ytrain = y[shuffler[:Ntrain]]
        Xtest = X[shuffler[Ntrain:]]
        ytest = y[shuffler[Ntrain:]]

        for spilt in range(num_splits):
            train_size = int(N * (spilt + 1) / num_splits)

            Xtrain_current = Xtrain[:train_size]
            ytrain_current = ytrain[:train_size]

            nbc.fit(Xtrain_current, ytrain_current)
            lr.fit(Xtrain_current, ytrain_current)

            yhat_nbc = nbc.predict(Xtest)
            yhat_lr  = lr.predict(Xtest)

            err_nbc = np.mean(yhat_nbc != ytest)
            err_lr  = np.mean(yhat_lr != ytest)

            all_nbc_errors[run, spilt] = err_nbc
            all_lr_errors[run, spilt] = err_lr

    tst_errs_nbc = np.mean(all_nbc_errors, axis=0)
    tst_errs_lr = np.mean(all_lr_errors, axis=0)
    
    return tst_errs_nbc, tst_errs_lr

    ###################################################
    ##### YOUR CODE ENDS HERE #########################
    ###################################################

In [ ]:
#iris
iris = load_iris()
X, y = iris['data'], iris['target']
compare = compareNBCvsLR(NBC(feature_types=['r', 'r', 'r', 'r']), LogisticRegression(max_iter=1000), X, y)


In [ ]:
def makePlot(nbc_perf, lr_perf, title=None, num_splits=10):
    fig = plt.figure()
    ax = fig.add_subplot(1, 1, 1)

    ax.tick_params(axis='both', labelsize=20)

    ax.set_xlabel('Percent of training data used', fontsize=20)
    ax.set_ylabel('Classification Error', fontsize=20)
    if title is not None: ax.set_title(title, fontsize=25)

    xaxis_scale = [(i + 1) * (100/num_splits) for i in range(num_splits)]
    plt.plot(xaxis_scale, nbc_perf, label='Naive Bayes')
    plt.plot(xaxis_scale, lr_perf, label='Logistic Regression', linestyle='dashed')
    
    ax.legend(loc='upper right', fontsize=20)
    plt.show()

In [ ]:
makePlot(compare[0], compare[1])


### Datasets

Tasks: For each dataset,
1. Prepare the data for the two classifiers, e.g., handle missing values and the categorical data. When you handle the categorical data, you should check whether the data is ordinal or not. If the data is ordinal, you should encode the data as integers. If the data is not ordinal, you should encode the data as one-hot vectors.
2. Show the first 5 rows of the prepared data
3. Compare the two classifiers on the dataset and generate the plots

The grading will be based on whether the data is correctly prepared and the plots are generated without errors. The grading will not be based on the performance of the classifiers and whether the plots are the same as in the paper. 

**Dataset 1: Iris Dataset**

https://scikit-learn.org/stable/auto_examples/datasets/plot_iris_dataset.html

In [ ]:
# TODO: insert your code for experiments
###################################################
##### YOUR CODE STARTS HERE #######################
###################################################
from sklearn.datasets import load_iris


# Load data
iris = load_iris()
X, y = iris['data'], iris['target']

# Run comparison
compare_iris = compareNBCvsLR(
    NBC(feature_types=['r', 'r', 'r', 'r']), 
    LogisticRegression(max_iter=1000), 
    X, y
)

# Generate the plot 
makePlot(compare_iris[0], compare_iris[1], title='Iris Dataset (Continuous)')


###################################################
##### YOUR CODE ENDS HERE #########################
###################################################

**Dataset 2: Voting Dataset**

https://archive.ics.uci.edu/ml/datasets/congressional+voting+records

The logistic regression line meets the naive bayes line early in the plot. 

In [ ]:
# load the dataset
# TODO: insert your code for experiments
###################################################
##### YOUR CODE STARTS HERE #######################
###################################################

voting = pd.read_csv('./data/voting.csv')

# Handle missing values (empty strings shown as '')
voting = voting.replace('', np.nan)
voting = voting.dropna()

# Encode labels: democrat=0, republican=1
voting['label'] = voting['label'].map({'democrat': 0, 'republican': 1})

# Encode features: y=1, n=0
feature_columns = voting.columns[1:]
voting[feature_columns] = voting[feature_columns].replace({'y': 1, 'n': 0})

# Prepare X and y
X = voting[feature_columns].values.astype(int)
y = voting['label'].values

# print(voting.head())
# print(f"\nX shape: {X.shape}, y shape: {y.shape}")

# Run comparison
compare_voting = compareNBCvsLR(
    NBC(feature_types=['b'] * 16), 
    LogisticRegression(max_iter=1000), 
    X, y
)

# Generate the plot
makePlot(compare_voting[0], compare_voting[1], title='Voting Dataset (Binary)')

###################################################
##### YOUR CODE ENDS HERE #########################
###################################################

**Dataset 3: Breast Cancer Dataset (Bonus Tasks)**

https://archive.ics.uci.edu/ml/datasets/breast+cancer

The dataset has continues, binary and categorical features. It also has missing values.

Hints: You can precompute the size of the domains of the categorical features.

In [ ]:
# load the dataset
# TODO: insert your code for experiments
###################################################
##### YOUR CODE STARTS HERE #######################
###################################################

cancer = pd.read_csv('./data/breast-cancer.csv')

# Handle missing values (marked as '?')
cancer = cancer.replace('?', np.nan)
cancer = cancer.dropna()

# Encode label: no-recurrence-events=0, recurrence-events=1
cancer['Class'] = cancer['Class'].map({'no-recurrence-events': 0, 'recurrence-events': 1})

# Encode all categorical features as integers
feature_columns = cancer.columns[1:]  # All columns except 'Class'
X_encoded = cancer[feature_columns].copy()

for col in feature_columns:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))

X = X_encoded.values
y = cancer['Class'].values

# print(cancer.head())
# print(f"\nX shape: {X.shape}, y shape: {y.shape}")

# Run comparison
compare_cancer = compareNBCvsLR(
    NBC(feature_types=['c'] * 9), 
    LogisticRegression(max_iter=1000), 
    X, y
)

# Generate the plot
makePlot(compare_cancer[0], compare_cancer[1], title='Breast Cancer Dataset (Categorical)')


###################################################
##### YOUR CODE ENDS HERE #########################
###################################################